# NB to check the data

# Plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.ticker as ticker
import os
import re
import numpy as np


xlabel_dict = {
    "time": "Waiting time in minutes",
    "pain": "Pain signal in percentage",
    "distance": "Distance to walk in kilometers",
    "hunger": "Waiting time in minutes",
}


def make_heat_maps(df, model_name, prompt, plot_dir, log_scale=True):
    df = df.copy()

    df["output_numeric"] = df["output"].map(
        lambda x: (
            1
            if re.search(r"\byes\b", x.strip(), flags=re.IGNORECASE)
            else (0 if re.search(r"\bno\b", x.strip(), flags=re.IGNORECASE) else np.nan)
        )
    )

    reward_values = np.sort(df["reward_value"].unique())
    quants = np.sort(df["quantity"].unique())

    experiment_arrays = []
    for exp, group in df.groupby("experiment"):
        pivot = group.pivot(
            index="reward_value", columns="quantity", values="output_numeric"
        )
        pivot = pivot.reindex(index=reward_values, columns=quants)
        pivot = pivot.sort_index().sort_index(axis=1)
        pivot = pivot.fillna(0)
        experiment_arrays.append(pivot.values)

    stacked = np.stack(experiment_arrays, axis=0)
    mean_array = np.mean(stacked, axis=0)
    quant_edges = compute_edges(quants)
    reward_edges = compute_edges(reward_values, log_scale=log_scale)

    XX, YY = np.meshgrid(quant_edges, reward_edges)

    plt.figure(figsize=(8, 6))
    plt.pcolormesh(XX, YY, mean_array, shading="auto", cmap="viridis", vmin=0, vmax=1)
    plt.colorbar(label="Probability of Yes")

    plt.xticks(quants[::2], labels=np.rint(quants[::2]).astype(int))
    plt.xlabel(xlabel_dict[prompt])

    ax = plt.gca()
    if log_scale:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(ticker.LogLocator(base=10, numticks=6))
        ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation(base=10))

        ax.set_ylabel("Money offered in euros (log scale)")
    else:
        ax.set_yticks(reward_values[::5])
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
        ax.set_ylabel("Money offered in euros")

    plt.title(model_name)

    os.makedirs(plot_dir, exist_ok=True)

    plot_file = f"{plot_dir}/{model_name}{'_log' if log_scale else ''}.png"
    plt.savefig(plot_file, dpi=300)
    plt.close()


def compute_edges(centers, log_scale=False):
    """
    Given a sorted array of center points, compute edges for pcolormesh.
    If log_scale is True, edges are computed in log space.
    """
    centers = np.asarray(centers)

    # Special case: Only one center
    if len(centers) == 1:
        c = centers[0]
        if log_scale:
            # Choose a factor for "half-bin" in log space, e.g. sqrt(10).
            return np.array([c / 10**0.5, c * 10**0.5])
        else:
            return np.array([c - 0.5, c + 0.5])

    if not log_scale:
        half_diffs = np.diff(centers) / 2.0
        edges = np.empty(len(centers) + 1)
        edges[0] = centers[0] - half_diffs[0]
        edges[-1] = centers[-1] + half_diffs[-1]
        edges[1:-1] = centers[:-1] + half_diffs
        return edges
    else:
        log_c = np.log(centers)
        half_diffs = np.diff(log_c) / 2.0
        log_edges = np.empty(len(centers) + 1)
        log_edges[0] = log_c[0] - half_diffs[0]
        log_edges[-1] = log_c[-1] + half_diffs[-1]
        log_edges[1:-1] = log_c[:-1] + half_diffs
        return np.exp(log_edges)

In [15]:
def combine_charts(plot_dir, prompt):
    """
    Combine chart images stored in a given directory into a composite figure.
    """
    file_list = sorted(
        [
            os.path.join(plot_dir, fname)
            for fname in os.listdir(plot_dir)
            if fname.lower().endswith(".png")
        ]
    )

    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
    axes_flat = axes.flatten()

    for i, filepath in enumerate(file_list):
        img = mpimg.imread(filepath)
        axes_flat[i].imshow(img)
        model_name = os.path.splitext(os.path.basename(filepath))[0]
        axes_flat[i].set_title(model_name)
        axes_flat[i].axis("off")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.suptitle(f"Heatmaps of {prompt}/money trade off scenario", fontsize=20)
    plt.tight_layout()

    save_file = f"{plot_dir}/{prompt}_heatmaps_combined_log.png"
    plt.savefig(save_file, dpi=300)
    print(f"Combined chart saved to {save_file}")
    plt.close(fig)

In [43]:
import pickle

prompt = "cot"

# path_results = f"results/{prompt}/vm_new_21x21x5/data_log.pkl"
# path_figures = f"figures/{prompt}/vm_new_21x21x5"

path = f"results/{prompt}/robus_1x101x5/data_log.pkl"

models = [
    "gpt4o",
    "claude3_5",
    "llama3_3_70b",
    "deepseek_v3",
    "mixtral8x22b",
    "gemini2",
]

with open(path, "rb") as f:
    data = pickle.load(f)

data

{'gpt4o':      reward_value  quantity  experiment  \
 0        0.100000      60.0           1   
 1        0.109648      60.0           1   
 2        0.120226      60.0           1   
 3        0.131826      60.0           1   
 4        0.144544      60.0           1   
 ..            ...       ...         ...   
 500    691.830971      60.0           5   
 501    758.577575      60.0           5   
 502    831.763771      60.0           5   
 503    912.010839      60.0           5   
 504   1000.000000      60.0           5   
 
                                                 output  
 0    In making this decision, I'll consider both th...  
 1    In this scenario, you have to balance the time...  
 2    In making this decision, I need to consider th...  
 3    To make an informed decision, let's consider t...  
 4    In this scenario, the user has the option to w...  
 ..                                                 ...  
 500  In making the decision, several factors are co...

In [44]:
for i, model in enumerate(models):
    if model in data.keys():
        print(f"{i}: {model}:\n\tno. of rows: {len(data[model])}")

0: gpt4o:
	no. of rows: 505
1: claude3_5:
	no. of rows: 505
2: llama3_3_70b:
	no. of rows: 505
3: deepseek_v3:
	no. of rows: 505
4: mixtral8x22b:
	no. of rows: 505
5: gemini2:
	no. of rows: 505


In [21]:
for model_name in models:
    df = data[model_name]
    make_heat_maps(df, model_name, prompt, path_figures)

combine_charts(path_figures, prompt)

Combined chart saved to figures/time/vm_new_21x21x5/time_heatmaps_combined_log.png


# Check the responses

In [45]:
import pickle


def get_multi_word_outputs(pkl_path):
    """
    Load a pickle of {model_name: DataFrame}, and return a dict of
    {model_name: filtered DataFrame} where 'output' has more than one word.

    A “word” here is just anything separated by whitespace.

    Parameters
    ----------
    pkl_path : str or Path
        Path to your .pkl file.

    Returns
    -------
    Dict[str, pandas.DataFrame]
    """
    # 1) load
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    # 2) for each model, filter rows where output has > 1 word
    result = {}
    for model_name, df in data.items():
        # ensure it's str, strip whitespace, split on whitespace, count >1
        mask = df["output"].astype(str).str.strip().str.split().str.len() > 1
        result[model_name] = df[mask].copy()

    return result

In [46]:
# with open(path_check, "rb") as f:
#     data = pickle.load(f)

data

{'gpt4o':      reward_value  quantity  experiment  \
 0        0.100000      60.0           1   
 1        0.109648      60.0           1   
 2        0.120226      60.0           1   
 3        0.131826      60.0           1   
 4        0.144544      60.0           1   
 ..            ...       ...         ...   
 500    691.830971      60.0           5   
 501    758.577575      60.0           5   
 502    831.763771      60.0           5   
 503    912.010839      60.0           5   
 504   1000.000000      60.0           5   
 
                                                 output  
 0    In making this decision, I'll consider both th...  
 1    In this scenario, you have to balance the time...  
 2    In making this decision, I need to consider th...  
 3    To make an informed decision, let's consider t...  
 4    In this scenario, the user has the option to w...  
 ..                                                 ...  
 500  In making the decision, several factors are co...

In [47]:
for i, model in enumerate(models):
    if model in data.keys():
        print(f"{i}: {model}:\n\tno. of rows: {len(data[model])}")

0: gpt4o:
	no. of rows: 505
1: claude3_5:
	no. of rows: 505
2: llama3_3_70b:
	no. of rows: 505
3: deepseek_v3:
	no. of rows: 505
4: mixtral8x22b:
	no. of rows: 505
5: gemini2:
	no. of rows: 505


In [48]:
filtered = get_multi_word_outputs(path)
for model, subdf in filtered.items():
    print(f"--- {model} ---")
    print(subdf)

--- gpt4o ---
     reward_value  quantity  experiment  \
0        0.100000      60.0           1   
1        0.109648      60.0           1   
2        0.120226      60.0           1   
3        0.131826      60.0           1   
4        0.144544      60.0           1   
..            ...       ...         ...   
500    691.830971      60.0           5   
501    758.577575      60.0           5   
502    831.763771      60.0           5   
503    912.010839      60.0           5   
504   1000.000000      60.0           5   

                                                output  
0    In making this decision, I'll consider both th...  
1    In this scenario, you have to balance the time...  
2    In making this decision, I need to consider th...  
3    To make an informed decision, let's consider t...  
4    In this scenario, the user has the option to w...  
..                                                 ...  
500  In making the decision, several factors are co...  
501  To make 

# to investigate

Check:
1. mixtral:
    * hunger - 398 (yes), 1759 (yes)
    * time - 256 (No), 447(yes), 457 (yes), 843 (no), 1485 (yes), 1783 (?), 2074 (yes)
2. pain prompt to check

In [106]:
prompt_check = "dutch"
path_check = f"results/{prompt_check}/robus_1x101x5/data_log.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)

In [108]:
check = data["mixtral8x22b"]["output"].iloc[325]  # 259 # 160
print(check)

''Nee''


In [105]:
import re
import numpy as np

result = (
    lambda x: (
        1
        if (
            re.search(r"answer", str(x or ""), re.IGNORECASE)
            and re.search(r"yes\b", str(x or ""), re.IGNORECASE)
        )
        else (
            0
            if (
                re.search(r"answer", str(x or ""), re.IGNORECASE)
                and re.search(r"no\b", str(x or ""), re.IGNORECASE)
            )
            else np.nan
        )
    )
)(check)

result

nan

# LLAMAAPI check

In [1]:
import json
from llamaapi import LlamaAPI

# Initialize the SDK
llama = LlamaAPI("5cb11816-d5b5-4978-bf19-76fc66c78ece")

# Build the API request
api_request_json = {
    "model": "llama3.1-70b",
    "messages": [
        {"role": "user", "content": "What is the weather like in Boston?"},
    ],
    "functions": [
        {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "days": {
                        "type": "number",
                        "description": "for how many days ahead you wants the forecast",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
            },
            "required": ["location", "days"],
        }
    ],
    "stream": False,
    "function_call": "get_current_weather",
}

# Execute the Request
response = llama.run(api_request_json)
print(json.dumps(response.json(), indent=2))

{
  "created": 1745221368,
  "model": "llama3.1-70b",
  "usage": {
    "prompt_tokens": 43,
    "completion_tokens": 239,
    "total_tokens": 282
  },
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "I'm not able to provide real-time weather information. However, I can give you general information about Boston's climate.\n\nBoston, Massachusetts has a humid continental climate with cold winters and warm summers. The city experiences a significant amount of precipitation throughout the year, with an average annual snowfall of around 43 inches.\n\nIn the spring (March to May), Boston's weather is typically mild, with temperatures ranging from the mid-40s to mid-60s Fahrenheit (7-18\u00b0C).\n\nIn the summer (June to August), the weather is warm and humid, with temperatures often reaching the mid-70s to mid-80s Fahrenheit (23-30\u00b0C).\n\nIn the fall (September to November), the weather is generally cool

# LR check

In [15]:
import pickle
import numpy as np

In [16]:
prompt_check = "time"
path_check = f"results/{prompt_check}/vm_new_21x21x5/LR_results.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)


data

{'gpt4o':     quantity  transition
 0        0.0  889.694330
 1       30.0    2.746645
 2       60.0    6.206626
 3       90.0    7.884002
 4      120.0   11.508410
 5      150.0   15.402263
 6      180.0   28.947751
 7      210.0   38.818728
 8      240.0   50.393398
 9      270.0   52.886127
 10     300.0   66.208759
 11     330.0   64.733511
 12     360.0  123.672645
 13     390.0  152.337848
 14     420.0  184.683592
 15     450.0  201.533459
 16     480.0  129.643889
 17     510.0  275.160364
 18     540.0  170.424769
 19     570.0  329.750985
 20     600.0  161.710978,
 'claude3_5':     quantity   transition
 0        0.0    30.885521
 1       30.0     4.363591
 2       60.0    14.793808
 3       90.0    17.956605
 4      120.0    44.149246
 5      150.0   111.970549
 6      180.0    57.623613
 7      210.0   202.750380
 8      240.0   195.125074
 9      270.0   279.215875
 10     300.0   121.508742
 11     330.0   278.661707
 12     360.0   190.887242
 13     390.0   238.057409


In [17]:
# prompt_check = "distance"
path_check = f"results/vm_new_21x21x5_tradeoff_table.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)


data

,model_name,distance|4.5,pain|50,time|60,hunger|60
0,claude3_5,4.48,15.22,14.79,145.42
1,deepseek_v3,3.88,22.99,4.49,33.44
2,gemini2,3.69,2.2,0.16,6.01
3,gpt4o,27.44,521.64,6.21,32.56
4,llama3_3_70b,1.01,1.01,1.01,79.32
5,mixtral8x22b,<0.1,NaN:y=0,23.42,<0.1


# LOG FIT CHECK

In [ ]:
10 ** (0.56)

3.630780547701014

In [72]:
# prompt_check = "distance"
path_check = f"results/distance/vm_1x101x5/data_log.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)

data["mixtral8x22b"]

,reward_value,quantity,experiment,output
0,0.100000,5.0,1,No.
1,0.109648,5.0,1,No.
2,0.120226,5.0,1,No.
3,0.131826,5.0,1,No.
4,0.144544,5.0,1,No.
...,...,...,...,...
500,691.830971,5.0,5,Yes.
501,758.577575,5.0,5,Yes.
502,831.763771,5.0,5,Yes.
503,912.010839,5.0,5,Yes.


In [73]:
data["mixtral8x22b"]["reward_value"].unique()

array([1.00000000e-01, 1.09648000e-01, 1.20226000e-01, 1.31826000e-01,
       1.44544000e-01, 1.58489000e-01, 1.73780000e-01, 1.90546000e-01,
       2.08930000e-01, 2.29087000e-01, 2.51189000e-01, 2.75423000e-01,
       3.01995000e-01, 3.31131000e-01, 3.63078000e-01, 3.98107000e-01,
       4.36516000e-01, 4.78630000e-01, 5.24807000e-01, 5.75440000e-01,
       6.30957000e-01, 6.91831000e-01, 7.58578000e-01, 8.31764000e-01,
       9.12011000e-01, 1.00000000e+00, 1.09647800e+00, 1.20226400e+00,
       1.31825700e+00, 1.44544000e+00, 1.58489300e+00, 1.73780100e+00,
       1.90546100e+00, 2.08929600e+00, 2.29086800e+00, 2.51188600e+00,
       2.75422900e+00, 3.01995200e+00, 3.31131100e+00, 3.63078100e+00,
       3.98107200e+00, 4.36515800e+00, 4.78630100e+00, 5.24807500e+00,
       5.75439900e+00, 6.30957300e+00, 6.91831000e+00, 7.58577600e+00,
       8.31763800e+00, 9.12010800e+00, 1.00000000e+01, 1.09647820e+01,
       1.20226440e+01, 1.31825670e+01, 1.44543980e+01, 1.58489320e+01,
      

In [ ]:
# prompt_check = "distance"
path_check = f"results/distance/vm_new_21x21x5/data_log.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)

data

import pandas as pd
from sklearn.linear_model import LogisticRegression
import re


def LR_transition_values(data, model_name, log=True):
    """
    Performs per-quantity logistic regression to find transition points
    for the given model_name. Rows with missing reward_value or output_binary
    are dropped. The function merges results into 'results/<prompt_type>/<experiment_name>/LR_results.pkl',
    keyed by model_name.
    Args:
        args: Command-line or config arguments (not directly used here,
              included for a consistent signature).
        data (dict): A dictionary {model_name: DataFrame} returned from your main experiment,
                     from which we extract data[model_name].
        model_name (str): Identifier for the model (e.g. "gpt4o").
    Returns:
        None. Loads and updates 'LR_results.pkl' so that LR_results[model_name] = DataFrame.
    """
    df = data[model_name].copy()
    df["output_binary"] = df["output"].map(
        lambda x: (
            1
            if re.search(r"\byes\b", x.strip(), flags=re.IGNORECASE)
            else (0 if re.search(r"\bno\b", x.strip(), flags=re.IGNORECASE) else np.nan)
        )
    )

    more_than_two_words = df["output"].str.split().str.len() > 2
    if more_than_two_words.any():
        print(
            "*" * 40,
            "\n[INFO] Found outputs with more than two words at these indices for: ",
            f"**{model_name}** model in prompt_type: **{args['prompt_type']}**: ",
        )
        for idx in df.index[more_than_two_words]:
            print(f"\t-index {idx}: '{df.loc[idx, 'output']}'")
        print("*" * 40)
        print(
            "[INFO] Consider manual checking of the classification of aforementioned index value in 'utils.LR_transition_values' function"
        )

    mask_nan = df["reward_value"].isna() | df["output_binary"].isna()
    df_nan = df[mask_nan]
    if not df_nan.empty:
        print(f"Dropping {len(df_nan)} rows due to NaN.\n")
    df = df.dropna(subset=["reward_value", "output_binary"])

    unique_quantities = sorted(df["quantity"].unique())
    results = []

    for q in unique_quantities:
        sub = df[df["quantity"] == q]

        X_raw = sub[["reward_value"]].astype(float)
        if log:
            if (X_raw <= 0).any().any():
                raise ValueError(
                    f"Found non‑positive reward_value(s) in quantity '{q}' while log_scale=True."
                )
            X = np.log10(X_raw)  # TODO check
        else:
            X = X_raw

        y = sub["output_binary"].astype(int)

        # Skip degenerate cases where y is constant
        if y.nunique() == 1:
            note_str = f"NaN:y={y.iloc[0]}"
            results.append((q, note_str))
            continue

        # Fit logistic regression
        model = LogisticRegression()
        model.fit(X, y)
        beta0 = model.intercept_[0]
        beta1 = model.coef_[0][0]

        if beta1 == 0:
            results.append((q, "Fit slope=0?"))
            continue

        # 50% acceptance => (beta0 + beta1*x)=0 => x*=-beta0/beta1
        x_star = -beta0 / beta1

        if log:
            x_star = 10**x_star  # TODO check

        results.append((q, x_star))

    results_df = pd.DataFrame(results, columns=["quantity", "transition"])
    return results_df

In [57]:
LR_transition_values(data, "mixtral8x22b")

,quantity,transition
0,0.0,0.0
1,1.5,0.218086
2,3.0,0.830203
3,4.5,3.622682
4,6.0,48.4051
5,7.5,1681.440051
6,9.0,2187.682907
7,10.5,551.878852
8,12.0,652191.233859
9,13.5,90284742635542872064.0


In [2]:
import pickle

path_check = f"results/vm_1x101x5_tradeoff_table.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)

data

,model_name,distance|5,pain|50,hunger|60,time|60
0,claude3_5,8.55,4.73,52.17,11.07
1,deepseek_v3,4.02,1.32,5.91,1.82
2,gemini2,2.55,1.83,2.14,0.20
3,gpt4o,22.29,145.55,23.12,5.29
4,llama3_3_70b,1.63,1.06,4.16,0.95
5,mixtral8x22b,2.82,NaN:y=0,<0.1,7.94


# COLORS

In [13]:
import re, numpy as np

# ── 1. your raw table values ───────────────────────────────────────────────
table_vals = [
    [8.55, 4.73, 52.17, 11.07, 19.13],
    [4.02, 1.32, 5.91, 1.82, 3.27],
    [2.55, 1.83, 2.14, 0.20, 1.68],
    [22.29, 145.55, 23.12, 5.29, 49.06],
    [1.63, 1.06, 4.16, 0.95, 1.95],
    [2.82, ">1000.00", "<0.10", 7.94, "252.72*"],
    [6.98, ">192.42*", "<14.60*", 4.55, ""],
]


# ── 2. utility to pull a float out of any cell ─────────────────────────────
def _num(x):
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        m = re.search(r"[\d.]+", x)
        if m:
            return float(m.group())
    return None


# global min/max across *numeric* entries
nums = [_num(c) for row in table_vals for c in row if _num(c) is not None]
VMIN, VMAX = min(nums), max(nums)

# ── 3. palette end-points (light green → light orange) ─────────────────────
LOW = np.array([199, 239, 199])  # #C7EFC7
HIGH = np.array([255, 215, 160])  # #FFD7A0


def _blend_rgb(t: float) -> np.ndarray:
    """linear interpolation t∈[0,1] from LOW to HIGH, returns [r,g,b] ints"""
    return ((1 - t) * LOW + t * HIGH).astype(int)


def rgb_to_html(rgb) -> str:
    """(r,g,b) 0-255 → '#rrggbb'"""
    return "#{:02X}{:02X}{:02X}".format(*rgb)


# ── 4. main converter ──────────────────────────────────────────────────────
def cell_to_html(cell):
    """Return an HTML colour for a table cell."""
    if cell == "":
        return "#FFFFFF"  # blank → white

    val = _num(cell)
    if val is None:  # non-numeric fallback
        return "#DCDCDC"  # light grey

    # markers forcing min/max colour
    if isinstance(cell, str) and cell.startswith("<"):
        val = VMIN
    elif isinstance(cell, str) and cell.startswith(">"):
        val = VMAX

    t = (val - VMIN) / (VMAX - VMIN) if VMAX > VMIN else 0
    return rgb_to_html(_blend_rgb(t))


# ── 5. build the table of HTML colours ─────────────────────────────────────
html_colors = [[cell_to_html(c) for c in row] for row in table_vals]

# pretty-print
for row, col_row in zip(table_vals, html_colors):
    print(row)
    print(col_row, "\n")

[8.55, 4.73, 52.17, 11.07, 19.13]
['#C7EEC6', '#C7EEC6', '#C9EDC4', '#C7EEC6', '#C8EEC6'] 

[4.02, 1.32, 5.91, 1.82, 3.27]
['#C7EEC6', '#C7EEC6', '#C7EEC6', '#C7EEC6', '#C7EEC6'] 

[2.55, 1.83, 2.14, 0.2, 1.68]
['#C7EEC6', '#C7EEC6', '#C7EEC6', '#C7EEC6', '#C7EEC6'] 

[22.29, 145.55, 23.12, 5.29, 49.06]
['#C8EEC6', '#CFEBC1', '#C8EEC6', '#C7EEC6', '#C9EDC5'] 

[1.63, 1.06, 4.16, 0.95, 1.95]
['#C7EEC6', '#C7EEC6', '#C7EEC6', '#C7EEC6', '#C7EEC6'] 

[2.82, '>1000.00', '<0.10', 7.94, '252.72*']
['#C7EEC6', '#FFD7A0', '#C7EFC7', '#C7EEC6', '#D5E8BD'] 

[6.98, '>192.42*', '<14.60*', 4.55, '']
['#C7EEC6', '#FFD7A0', '#C7EFC7', '#C7EEC6', '#FFFFFF'] 



In [15]:
import re, numpy as np, math

# ── 1. your data ------------------------------------------------------------
table_vals = [
    [8.55, 4.73, 52.17, 11.07, 19.13],
    [4.02, 1.32, 5.91, 1.82, 3.27],
    [2.55, 1.83, 2.14, 0.20, 1.68],
    [22.29, 145.55, 23.12, 5.29, 49.06],
    [1.63, 1.06, 4.16, 0.95, 1.95],
    [2.82, ">1000.00", "<0.10", 7.94, "252.72*"],
    [6.98, ">192.42*", "<14.60*", 4.55, ""],
]


# ── 2. extract numbers ------------------------------------------------------
def _num(x):
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        m = re.search(r"[\d.]+", x)
        if m:
            return float(m.group())
    return None


# gather numeric entries
numeric_vals = [_num(c) for row in table_vals for c in row if _num(c) is not None]

# smallest positive value (for log(0) safety)
EPS = min(v for v in numeric_vals if v > 0) * 0.1

# log-space min / max
VMIN_LOG = math.log10(min(numeric_vals))
VMAX_LOG = math.log10(max(numeric_vals))

# ── 3. palette endpoints (light green → light orange) -----------------------
LOW = np.array([199, 239, 199])  # #C7EFC7
HIGH = np.array([255, 215, 160])  # #FFD7A0


def _blend_rgb(t: float) -> np.ndarray:
    return ((1 - t) * LOW + t * HIGH).astype(int)


def rgb_to_hex(rgb) -> str:
    return "#{:02X}{:02X}{:02X}".format(*rgb)


# ── 4. log-scale colour for one cell ---------------------------------------
def cell_to_html_log(cell):
    if cell == "":  # blank → white
        return "#FFFFFF"

    val = _num(cell)
    if val is None:  # unparsable text
        return "#DCDCDC"

    # treat '<' values as low, '>' values as high
    if isinstance(cell, str) and cell.startswith("<"):
        val = min(numeric_vals)
    elif isinstance(cell, str) and cell.startswith(">"):
        val = max(numeric_vals)

    # safeguard against zero or negative
    val = max(val, EPS)
    t = (math.log10(val) - VMIN_LOG) / (VMAX_LOG - VMIN_LOG)
    return rgb_to_hex(_blend_rgb(t))


# ── 5. produce the HTML colour table ---------------------------------------
html_colors_log = [[cell_to_html_log(c) for c in row] for row in table_vals]

# ► demo print
for row, colors in zip(table_vals, html_colors_log):
    print(row)
    print(colors, "\n")

[8.55, 4.73, 52.17, 11.07, 19.13]
['#E2E3B4', '#DEE4B6', '#EDDEAC', '#E3E2B3', '#E6E1B0'] 

[4.02, 1.32, 5.91, 1.82, 3.27]
['#DDE5B7', '#D6E8BC', '#DFE4B5', '#D8E7BA', '#DCE5B8'] 

[2.55, 1.83, 2.14, 0.2, 1.68]
['#DAE6B9', '#D8E7BA', '#D9E7BA', '#CBEDC4', '#D8E7BB'] 

[22.29, 145.55, 23.12, 5.29, 49.06]
['#E7E0B0', '#F3DCA8', '#E8E0AF', '#DFE4B6', '#ECDEAC'] 

[1.63, 1.06, 4.16, 0.95, 1.95]
['#D7E7BB', '#D5E8BD', '#DDE5B7', '#D4E9BD', '#D9E7BA'] 

[2.82, '>1000.00', '<0.10', 7.94, '252.72*']
['#DBE6B8', '#FFD7A0', '#C7EFC7', '#E1E3B4', '#F6DAA5'] 

[6.98, '>192.42*', '<14.60*', 4.55, '']
['#E0E3B5', '#FFD7A0', '#C7EFC7', '#DEE5B6', '#FFFFFF'] 



In [29]:
import re

# ----- 1. raw table values --------------------------------------------------
table_vals = [
    [8.55, 4.73, 52.17, 11.07, 19.13],
    [4.02, 1.32, 5.91, 1.82, 3.27],
    [2.55, 1.83, 2.14, 0.20, 1.68],
    [22.29, 145.55, 23.12, 5.29, 49.06],
    [1.63, 1.06, 4.16, 0.95, 1.95],
    [2.82, ">1000.00", "<0.10", 7.94, "252.72*"],
    [6.98, ">192.42*", "<14.60*", 4.55, ""],
]

# ----- 2. bucket thresholds & colours --------------------------------------
THRESHOLDS = [20, 40, 60, 80]  # in percent of GLOBAL max
COLOURS = ["#FFFFFF", "#DBEFE2", "#BEE3C9", "#8ED0A0", "#63BE7B"]

SHADE_BLANK = "#FFFFFF"
SHADE_FALLBACK = "#FCFCFF"


# ----- 3. helper to extract number -----------------------------------------
def _num(x):
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        m = re.search(r"[\d.]+", x)
        if m:
            return float(m.group())
    return None


# overall max (ignore >, <, blanks)
numeric_vals = [_num(c) for row in table_vals for c in row if _num(c) is not None]
GLOBAL_MAX = max(numeric_vals)


# ----- 4. mapping function (global) ----------------------------------------
def cell_to_html(cell):
    if cell == "":
        return SHADE_BLANK

    if isinstance(cell, str) and cell.startswith("<"):
        return COLOURS[0]  # lightest
    if isinstance(cell, str) and cell.startswith(">"):
        return COLOURS[-1]  # darkest

    val = _num(cell)
    if val is None:
        return SHADE_FALLBACK

    pct = 100 * val / GLOBAL_MAX if GLOBAL_MAX > 0 else 0

    for thr, col in zip(THRESHOLDS, COLOURS):
        if pct < thr:
            return col
    return COLOURS[-1]


# ----- 5. build colour table -----------------------------------------------
html_colors = [[cell_to_html(c) for c in row] for row in table_vals]

# quick demo print
for raw, col in zip(table_vals, html_colors):
    print(raw)
    print(col, "\n")

[8.55, 4.73, 52.17, 11.07, 19.13]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[4.02, 1.32, 5.91, 1.82, 3.27]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[2.55, 1.83, 2.14, 0.2, 1.68]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[22.29, 145.55, 23.12, 5.29, 49.06]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[1.63, 1.06, 4.16, 0.95, 1.95]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[2.82, '>1000.00', '<0.10', 7.94, '252.72*']
['#FFFFFF', '#63BE7B', '#FFFFFF', '#FFFFFF', '#DBEFE2'] 

[6.98, '>192.42*', '<14.60*', 4.55, '']
['#FFFFFF', '#63BE7B', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 



In [35]:
import re

# ── 1. your raw table -------------------------------------------------------
table_vals = [
    [
        8.55,
        4.73,
        52.17,
        11.07,
    ],
    [
        4.02,
        1.32,
        5.91,
        1.82,
    ],
    [
        2.55,
        1.83,
        2.14,
        0.20,
    ],
    [
        22.29,
        145.55,
        23.12,
        5.29,
    ],
    [
        1.63,
        1.06,
        4.16,
        0.95,
    ],
    [2.82, ">1000.00", "<0.10", 7.94],
]


# ── 2. numeric helper -------------------------------------------------------
def _num(x):
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        m = re.search(r"[\d.]+", x)
        if m:
            return float(m.group())
    return None


numeric_vals = [_num(c) for row in table_vals for c in row if _num(c) is not None]
GLOBAL_MAX = max(numeric_vals)


# ── 3. build 200-shade palette (white → deep green) -------------------------
def lerp(a, b, t):  # linear interpolation helper
    return int(round((1 - t) * a + t * b))


deep_green = (14, 128, 68)  # RGB for #0E8044
palette = []
for i in range(6):
    t = i / 5  # 0 … 1
    r = lerp(255, deep_green[0], t)
    g = lerp(255, deep_green[1], t)
    b = lerp(255, deep_green[2], t)
    palette.append(f"#{r:02X}{g:02X}{b:02X}")

# palette[0]  = '#FFFFFF',  palette[199] = '#0E8044'


# ── 4. cell → HTML colour (global) -----------------------------------------
def cell_to_html(cell):
    # blank cell stays white
    if cell == "":
        return palette[0]

    # force extremes for "<" / ">" markers
    if isinstance(cell, str) and cell.startswith("<"):
        return palette[0]
    if isinstance(cell, str) and cell.startswith(">"):
        return palette[-1]

    val = _num(cell)
    if val is None:  # unparsable text
        return "#F0F0F0"

    pct = val / GLOBAL_MAX  # 0 … 1
    bucket = min(int(pct * 6), 5)  # 0 … 199
    return palette[bucket]


# ── 5. build the HTML-colour table -----------------------------------------
html_colors = [[cell_to_html(c) for c in row] for row in table_vals]

# quick demo print -----------------------------------------------------------
for raw, col in zip(table_vals, html_colors):
    print(raw)
    print(col, "\n")

[8.55, 4.73, 52.17, 11.07]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[4.02, 1.32, 5.91, 1.82]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[2.55, 1.83, 2.14, 0.2]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[22.29, 145.55, 23.12, 5.29]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[1.63, 1.06, 4.16, 0.95]
['#FFFFFF', '#FFFFFF', '#FFFFFF', '#FFFFFF'] 

[2.82, '>1000.00', '<0.10', 7.94]
['#FFFFFF', '#0E8044', '#FFFFFF', '#FFFFFF'] 



# BOOTStrapping

In [5]:
import pickle

prompt = "distance"
path_check = f"results/{prompt}/vm_1x101x5/data_log.pkl"

with open(path_check, "rb") as f:
    data = pickle.load(f)

In [6]:
data["claude3_5"]

,reward_value,quantity,experiment,output
0,0.100000,5.0,1,No
1,0.109648,5.0,1,No
2,0.120226,5.0,1,No
3,0.131826,5.0,1,No
4,0.144544,5.0,1,No
...,...,...,...,...
500,691.830971,5.0,5,Yes
501,758.577575,5.0,5,Yes
502,831.763771,5.0,5,Yes
503,912.010839,5.0,5,Yes


In [7]:
args = {
    "prompt_type": prompt,
    "log_scale_money": True,
    "money_min": -1,
    "money_max": 3,
    "output_dir": "results_bootstrap",
    "experiment_name": "bootstrap",
}

In [ ]:
import re
import numpy as np
from sklearn.linear_model import LogisticRegression
import pandas as pd


def LR_transition_values(args, data, model_name):
    """
    Performs per-quantity logistic regression to find transition points
    for the given model_name. Rows with missing reward_value or output_binary
    are dropped. The function merges results into 'results/<prompt_type>/<experiment_name>/LR_results.pkl',
    keyed by model_name.
    Args:
        args: Command-line or config arguments (not directly used here,
              included for a consistent signature).
        data (dict): A dictionary {model_name: DataFrame} returned from your main experiment,
                     from which we extract data[model_name].
        model_name (str): Identifier for the model (e.g. "gpt4o").
    Returns:
        None. Loads and updates 'LR_results.pkl' so that LR_results[model_name] = DataFrame.
    """
    df = data[model_name].copy()

    if args["prompt_type"] == "chinese":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if (
                    re.search(
                        r"^\s*是\s*(?:"
                        r"的"
                        r"|（.*?）"
                        r"|\(.*?\)"
                        r")?\s*[。.．!！?？]?\s*$",
                        str(x or "").strip(),
                    )
                    or re.search(r"\byes\b", str(x or "").strip(), flags=re.IGNORECASE)
                )
                else (
                    0
                    if (
                        re.search(r"^\s*否\s*[。.．!！?？]?\s*$", str(x or "").strip())
                        or re.search(
                            r"^\s*不(?:是)?\s*[。.．!！?？]?\s*$", str(x or "").strip()
                        )
                        or re.search(
                            r"\bno\b", str(x or "").strip(), flags=re.IGNORECASE
                        )
                    )
                    else np.nan
                )
            )
        )

    elif args["prompt_type"] == "french":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"^\s*oui\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"^\s*non\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[123, "output_binary"] = 0
            df.loc[451, "output_binary"] = 0
            df.loc[95, "output_binary"] = 1

    elif args["prompt_type"] == "dutch":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"^\s*ja\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"^\s*nee\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[1, "output_binary"] = 0
            df.loc[144, "output_binary"] = 0
            df.loc[271, "output_binary"] = 1
            df.loc[325, "output_binary"] = 0
            df.loc[468, "output_binary"] = 1

    elif args["prompt_type"] == "cot":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if (
                    re.search(r"answer", str(x or ""), re.IGNORECASE)
                    and re.search(r"yes\b", str(x or ""), re.IGNORECASE)
                )
                else (
                    0
                    if (
                        re.search(r"answer", str(x or ""), re.IGNORECASE)
                        and re.search(r"no\b", str(x or ""), re.IGNORECASE)
                    )
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[450, "output_binary"] = 1

    else:
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"\byes\b", x.strip(), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"\bno\b", x.strip(), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if args["prompt_type"] == "firstperson" and model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[166, "output_binary"] = 1

        if args["prompt_type"] == "man" and model_name == "llama3_3_70b":
            # Manual adjustments
            df.loc[494, "output_binary"] = 1

    more_than_two_words = df["output"].str.split().str.len() > 2
    if more_than_two_words.any() and args["prompt_type"] != "cot":
        print(
            "[INFO] Found outputs with more than two words at these indices for: ",
            f"**{model_name}** model in prompt_type: **{args['prompt_type']}**: ",
        )
        for idx in df.index[more_than_two_words]:
            print(f"\t-index {idx}: '{df.loc[idx, 'output']}'")
        print(
            "[INFO] Consider manual checking of the classification of aforementioned index value in 'utils.LR_transition_values' function."
        )

    mask_nan = df["reward_value"].isna() | df["output_binary"].isna()
    df_nan = df[mask_nan]
    if not df_nan.empty:
        print(
            f'\nModel: **{model_name}**, prompt type: **{args["prompt_type"]}**\n',
            df_nan,
        )
        print(f"\n[INFO] Dropping {len(df_nan)} rows due to NaN.\n")
    df = df.dropna(subset=["reward_value", "output_binary"])

    unique_quantities = sorted(df["quantity"].unique())
    results = []
    print("unique_quantities: ", unique_quantities)

    for q in unique_quantities:
        print("q: ", q)
        sub = df[df["quantity"] == q]
        print("sub: ", sub)

        X_raw = sub[["reward_value"]].astype(float)
        print("X_raw: ", X_raw)
        if args["log_scale_money"]:
            if (X_raw <= 0).any().any():
                raise ValueError(
                    f"Found non‑positive reward_value(s) in quantity '{q}' while log_scale=True."
                )
            X = np.log10(X_raw)
        else:
            X = X_raw
        y = sub["output_binary"].astype(int)

        # Skip degenerate cases where y is constant
        if y.nunique() == 1:
            if args["log_scale_money"]:
                if y.iloc[0] == 0:
                    note_str = f'>{10**args["money_max"]}'
                else:
                    note_str = f'<{10**args["money_min"]}'
            else:
                if y.iloc[0] == 0:
                    note_str = f'>{args["money_max"]}'
                else:
                    note_str = f'<{args["money_min"]}'

            results.append((q, note_str))
            continue

        # Fit logistic regression
        model = LogisticRegression()
        print("X: ", X)
        print("y: ", y)
        model.fit(X, y)
        beta0 = model.intercept_[0]
        beta1 = model.coef_[0][0]

        if beta1 == 0:
            results.append((q, "Fit slope=0?"))
            continue

        # 50% acceptance => (beta0 + beta1*x)=0 => x*=-beta0/beta1
        x_star = -beta0 / beta1

        if args["log_scale_money"]:
            x_star = 10**x_star

        results.append((q, x_star))
        print("*" * 30)

    results_df = pd.DataFrame(results, columns=["quantity", "transition"])

    pickle_filename = f'{args["output_dir"]}/{args["prompt_type"]}/{args["experiment_name"]}/LR_results.pkl'
    try:
        with open(pickle_filename, "rb") as f:
            LR_results = pickle.load(f)
    except FileNotFoundError:
        LR_results = {}

    LR_results[model_name] = results_df

    return results_df

In [13]:
LR_transition_values(args, data, "claude3_5")

unique_quantities:  [np.float64(5.0)]
q:  5.0
sub:       reward_value  quantity  experiment output  output_binary
0        0.100000       5.0           1     No              0
1        0.109648       5.0           1     No              0
2        0.120226       5.0           1     No              0
3        0.131826       5.0           1     No              0
4        0.144544       5.0           1     No              0
..            ...       ...         ...    ...            ...
500    691.830971       5.0           5    Yes              1
501    758.577575       5.0           5    Yes              1
502    831.763771       5.0           5    Yes              1
503    912.010839       5.0           5    Yes              1
504   1000.000000       5.0           5    Yes              1

[505 rows x 5 columns]
X_raw:       reward_value
0        0.100000
1        0.109648
2        0.120226
3        0.131826
4        0.144544
..            ...
500    691.830971
501    758.577575
502    831

,quantity,transition
0,5.0,8.548864


In [25]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.linear_model import LogisticRegression


def plot_LR_transitions(args, data, model_name, save_plots=True):
    """
    Plots the logistic regression results for each quantity, showing:
    1. Original data points
    2. Fitted sigmoid curve
    3. Transition value (where probability = 0.5)

    Args:
        args: Command-line or config arguments
        data (dict): A dictionary {model_name: DataFrame} from the main experiment
        model_name (str): Identifier for the model (e.g. "gpt4o")
        save_plots (bool): Whether to save the plots to disk (default: True)

    Returns:
        dict: Dictionary mapping quantity to the transition value
    """
    # Ensure output directory exists
    if save_plots and isinstance(args, dict):
        os.makedirs(
            f'{args["output_dir"]}/{args["prompt_type"]}/{args["experiment_name"]}',
            exist_ok=True,
        )
    df = data[model_name].copy()

    # Apply the same binary output classification as in LR_transition_values
    import re
    import numpy as np

    if args["prompt_type"] == "chinese":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if (
                    re.search(
                        r"^\s*是\s*(?:"
                        r"的"
                        r"|（.*?）"
                        r"|\(.*?\)"
                        r")?\s*[。.．!！?？]?\s*$",
                        str(x or "").strip(),
                    )
                    or re.search(r"\byes\b", str(x or "").strip(), flags=re.IGNORECASE)
                )
                else (
                    0
                    if (
                        re.search(r"^\s*否\s*[。.．!！?？]?\s*$", str(x or "").strip())
                        or re.search(
                            r"^\s*不(?:是)?\s*[。.．!！?？]?\s*$", str(x or "").strip()
                        )
                        or re.search(
                            r"\bno\b", str(x or "").strip(), flags=re.IGNORECASE
                        )
                    )
                    else np.nan
                )
            )
        )

    elif args["prompt_type"] == "french":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"^\s*oui\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"^\s*non\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[123, "output_binary"] = 0
            df.loc[451, "output_binary"] = 0
            df.loc[95, "output_binary"] = 1

    elif args["prompt_type"] == "dutch":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"^\s*ja\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"^\s*nee\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[1, "output_binary"] = 0
            df.loc[144, "output_binary"] = 0
            df.loc[271, "output_binary"] = 1
            df.loc[325, "output_binary"] = 0
            df.loc[468, "output_binary"] = 1

    elif args["prompt_type"] == "cot":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if (
                    re.search(r"answer", str(x or ""), re.IGNORECASE)
                    and re.search(r"yes\b", str(x or ""), re.IGNORECASE)
                )
                else (
                    0
                    if (
                        re.search(r"answer", str(x or ""), re.IGNORECASE)
                        and re.search(r"no\b", str(x or ""), re.IGNORECASE)
                    )
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[450, "output_binary"] = 1

    else:
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"\byes\b", str(x or "").strip(), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"\bno\b", str(x or "").strip(), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if args["prompt_type"] == "firstperson" and model_name == "mixtral8x22b":
            # Manual adjustments
            df.loc[166, "output_binary"] = 1

        if args["prompt_type"] == "man" and model_name == "llama3_3_70b":
            # Manual adjustments
            df.loc[494, "output_binary"] = 1

    # Check and report outputs with more than two words
    more_than_two_words = df["output"].str.split().str.len() > 2
    if more_than_two_words.any() and args["prompt_type"] != "cot":
        print(
            "[INFO] Found outputs with more than two words at these indices for: ",
            f"**{model_name}** model in prompt_type: **{args['prompt_type']}**: ",
        )
        for idx in df.index[more_than_two_words]:
            print(f"\t-index {idx}: '{df.loc[idx, 'output']}'")
        print(
            "[INFO] Consider manual checking of the classification of aforementioned index value."
        )

    # Drop rows with missing values
    mask_nan = df["reward_value"].isna() | df["output_binary"].isna()
    df_nan = df[mask_nan]
    if not df_nan.empty:
        print(
            f'\nModel: **{model_name}**, prompt type: **{args["prompt_type"]}**\n',
            df_nan,
        )
        print(f"\n[INFO] Dropping {len(df_nan)} rows due to NaN.\n")

    df = df.dropna(subset=["reward_value", "output_binary"])

    unique_quantities = sorted(df["quantity"].unique())
    results = {}

    # Create a figure with subplots if there are multiple quantities
    n_quantities = len(unique_quantities)
    fig_cols = min(3, n_quantities)
    fig_rows = (n_quantities + fig_cols - 1) // fig_cols

    if save_plots:
        fig, axes = plt.subplots(
            fig_rows, fig_cols, figsize=(5 * fig_cols, 4 * fig_rows)
        )
        if n_quantities == 1:
            axes = np.array([axes])
        axes = axes.flatten()

    for i, q in enumerate(unique_quantities):
        sub = df[df["quantity"] == q]

        X_raw = sub[["reward_value"]].astype(float)

        # Apply log scaling if specified
        if args.get("log_scale_money", False) or (
            isinstance(args, dict) and args.get("log_scale_money", False)
        ):
            if (X_raw <= 0).any().any():
                print(
                    f"Warning: Found non-positive reward_value(s) in quantity '{q}' while log_scale=True."
                )
                continue
            X = np.log10(X_raw)
            x_label = "Log10(Reward Value)"
        else:
            X = X_raw
            x_label = "Reward Value"

        y = sub["output_binary"].astype(int)

        # Skip degenerate cases where y is constant
        if y.nunique() == 1:
            transition = "N/A - All responses are the same"
            results[q] = transition
            continue

        # Fit logistic regression
        model = LogisticRegression()
        model.fit(X, y)
        beta0 = model.intercept_[0]
        beta1 = model.coef_[0][0]

        if beta1 == 0:
            transition = "N/A - Fitted slope is 0"
            results[q] = transition
            continue

        # Calculate transition point (50% probability)
        x_star = -beta0 / beta1

        # Generate points for plotting sigmoid curve
        x_min, x_max = X["reward_value"].min(), X["reward_value"].max()
        x_padding = (x_max - x_min) * 0.1
        x_range = np.linspace(x_min - x_padding, x_max + x_padding, 1000).reshape(-1, 1)
        y_proba = model.predict_proba(x_range)[:, 1]

        # Convert transition point back from log scale if needed
        if args.get("log_scale_money", False) or (
            isinstance(args, dict) and args.get("log_scale_money", False)
        ):
            display_x_star = 10**x_star
            transition_label = f"Transition: ${display_x_star:.2f}"
            # Also convert x_range for plotting
            x_range_display = 10**x_range
        else:
            display_x_star = x_star
            transition_label = f"Transition: ${display_x_star:.2f}"
            x_range_display = x_range

        results[q] = display_x_star

        if save_plots:
            ax = axes[i]

            # Plot original data points (jittered on y-axis for better visualization)
            jitter = np.random.uniform(-0.05, 0.05, size=len(y))
            ax.scatter(X_raw, y + jitter, alpha=0.6, color="blue", label="Data Points")

            # Plot sigmoid curve
            if args.get("log_scale_money", False) or (
                isinstance(args, dict) and args.get("log_scale_money", False)
            ):
                ax.plot(10**x_range, y_proba, color="red", label="Fitted Sigmoid")
                ax.set_xscale("log")
            else:
                ax.plot(x_range, y_proba, color="red", label="Fitted Sigmoid")

            # Plot transition point
            ax.axvline(
                x=display_x_star, color="green", linestyle="--", label=transition_label
            )

            # Add probability = 0.5 line
            ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.7)

            # Labels and title
            ax.set_xlabel(x_label)
            ax.set_ylabel("P(Yes)")
            ax.set_title(
                f'Model: {model_name}, prompt: {args["prompt_type"]}, quantity: {q}'
            )
            ax.set_ylim(-0.1, 1.1)
            ax.legend(loc="best")

            # Set x-axis to a more readable format
            if not args.get("log_scale_money", False):
                ax.ticklabel_format(style="plain", axis="x")

    if save_plots:
        # Hide unused subplots
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        # Save figure
        if isinstance(args, dict):
            plot_path = (
                f'{args["output_dir"]}/{args["prompt_type"]}/LR_plots_{model_name}.png'
            )
            plt.tight_layout()
            plt.savefig(plot_path, dpi=300)
            print(f"Saved plots to {plot_path}")
        else:
            plt.tight_layout()
            plt.show()  # Just show the plot if args is not a dict
        plt.close()

        print(f"Saved plots to {plot_path}")

    return results

In [26]:
plot_LR_transitions(args, data, "claude3_5")

/home/mateuszcedro/mateuszcedro/PhD/repos/llm_trade_offs/LLM-tradeoffs/venv_llm_to/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


Saved plots to results_bootstrap/distance/LR_plots_claude3_5.png
Saved plots to results_bootstrap/distance/LR_plots_claude3_5.png


{np.float64(5.0): np.float64(8.548864049758844)}

In [40]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample


def bootstrap_LR_analysis(args, data, model_name, n_bootstrap=20, save_plots=True):
    """
    Performs bootstrap resampling on the data and fits logistic regression models
    to each bootstrap sample, calculating transition values and confidence intervals.

    Args:
        args: Command-line or config arguments
        data (dict): A dictionary {model_name: DataFrame} from the main experiment
        model_name (str): Identifier for the model (e.g. "gpt4o")
        n_bootstrap (int): Number of bootstrap samples to generate
        save_plots (bool): Whether to save plots to disk

    Returns:
        dict: Dictionary with transition values and confidence intervals for each quantity
    """
    # Ensure output directory exists if we're saving plots
    if save_plots and isinstance(args, dict):
        os.makedirs(
            f'{args["output_dir"]}/{args["prompt_type"]}/{args["experiment_name"]}',
            exist_ok=True,
        )

    df = data[model_name].copy()

    # Apply binary output classification (same as in LR_transition_values)
    import re
    import numpy as np

    # Binary classification logic from the original function
    if args["prompt_type"] == "chinese":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if (
                    re.search(
                        r"^\s*是\s*(?:"
                        r"的"
                        r"|（.*?）"
                        r"|\(.*?\)"
                        r")?\s*[。.．!！?？]?\s*$",
                        str(x or "").strip(),
                    )
                    or re.search(r"\byes\b", str(x or "").strip(), flags=re.IGNORECASE)
                )
                else (
                    0
                    if (
                        re.search(r"^\s*否\s*[。.．!！?？]?\s*$", str(x or "").strip())
                        or re.search(
                            r"^\s*不(?:是)?\s*[。.．!！?？]?\s*$", str(x or "").strip()
                        )
                        or re.search(
                            r"\bno\b", str(x or "").strip(), flags=re.IGNORECASE
                        )
                    )
                    else np.nan
                )
            )
        )
    elif args["prompt_type"] == "french":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"^\s*oui\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"^\s*non\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            if 123 in df.index:
                df.loc[123, "output_binary"] = 0
            if 451 in df.index:
                df.loc[451, "output_binary"] = 0
            if 95 in df.index:
                df.loc[95, "output_binary"] = 1
    elif args["prompt_type"] == "dutch":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"^\s*ja\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"^\s*nee\.?\s*$", str(x or ""), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            if 1 in df.index:
                df.loc[1, "output_binary"] = 0
            if 144 in df.index:
                df.loc[144, "output_binary"] = 0
            if 271 in df.index:
                df.loc[271, "output_binary"] = 1
            if 325 in df.index:
                df.loc[325, "output_binary"] = 0
            if 468 in df.index:
                df.loc[468, "output_binary"] = 1
    elif args["prompt_type"] == "cot":
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if (
                    re.search(r"answer", str(x or ""), re.IGNORECASE)
                    and re.search(r"yes\b", str(x or ""), re.IGNORECASE)
                )
                else (
                    0
                    if (
                        re.search(r"answer", str(x or ""), re.IGNORECASE)
                        and re.search(r"no\b", str(x or ""), re.IGNORECASE)
                    )
                    else np.nan
                )
            )
        )
        if model_name == "mixtral8x22b":
            # Manual adjustments
            if 450 in df.index:
                df.loc[450, "output_binary"] = 1
    else:
        df["output_binary"] = df["output"].map(
            lambda x: (
                1
                if re.search(r"\byes\b", str(x or "").strip(), flags=re.IGNORECASE)
                else (
                    0
                    if re.search(r"\bno\b", str(x or "").strip(), flags=re.IGNORECASE)
                    else np.nan
                )
            )
        )
        if args["prompt_type"] == "firstperson" and model_name == "mixtral8x22b":
            # Manual adjustments
            if 166 in df.index:
                df.loc[166, "output_binary"] = 1

        if args["prompt_type"] == "man" and model_name == "llama3_3_70b":
            # Manual adjustments
            if 494 in df.index:
                df.loc[494, "output_binary"] = 1

    # Drop rows with missing values
    df = df.dropna(subset=["reward_value", "output_binary"])

    # Get unique quantities
    unique_quantities = sorted(df["quantity"].unique())
    bootstrap_results = {}

    # For each quantity, perform bootstrap analysis
    for q in unique_quantities:
        sub_df = df[df["quantity"] == q]

        # Skip quantities with too few data points
        if len(sub_df) < 5:
            print(
                f"Skipping quantity '{q}' due to insufficient data points ({len(sub_df)} available)"
            )
            continue

        # Skip degenerate cases where all responses are the same
        if sub_df["output_binary"].nunique() == 1:
            print(f"Skipping quantity '{q}' because all responses are the same")
            continue

        # Store all bootstrap transitions for this quantity
        transitions = []

        # Create bootstrap samples and fit models
        seed = 42
        np.random.seed(seed)
        for i in range(n_bootstrap):
            # Create bootstrap sample with replacement
            bootstrap_sample = resample(sub_df, replace=True, n_samples=len(sub_df))

            # Skip if the bootstrap sample is degenerate (all yes or all no)
            if bootstrap_sample["output_binary"].nunique() == 1:
                continue

            # Prepare data for logistic regression
            X_raw = bootstrap_sample[["reward_value"]].astype(float)

            # Apply log scaling if specified
            if args.get("log_scale_money", False) or (
                isinstance(args, dict) and args.get("log_scale_money", False)
            ):
                if (X_raw <= 0).any().any():
                    continue  # Skip bootstrap samples with non-positive values in log scale
                X = np.log10(X_raw)
            else:
                X = X_raw

            y = bootstrap_sample["output_binary"].astype(int)

            # Fit logistic regression
            try:
                model = LogisticRegression(
                    max_iter=1000
                )  # Increase max_iter to ensure convergence
                model.fit(X, y)

                beta0 = model.intercept_[0]
                beta1 = model.coef_[0][0]

                # Skip if the slope is too close to zero (flat line)
                if abs(beta1) < 1e-6:
                    continue

                # Calculate transition point (-beta0/beta1)
                x_star = -beta0 / beta1

                # Convert from log scale if needed
                if args.get("log_scale_money", False) or (
                    isinstance(args, dict) and args.get("log_scale_money", False)
                ):
                    x_star = 10**x_star

                transitions.append(x_star)
            except Exception as e:
                print(f"Error fitting bootstrap sample for quantity '{q}': {e}")
                continue

        # Calculate statistics from bootstrap samples
        if transitions:
            transitions = np.array(transitions)
            mean_transition = np.mean(transitions)
            median_transition = np.median(transitions)
            ci_low = np.percentile(transitions, 2.5)  # 2.5th percentile for 95% CI
            ci_high = np.percentile(transitions, 97.5)  # 97.5th percentile for 95% CI

            bootstrap_results[q] = {
                "transitions": transitions,
                "mean": mean_transition,
                "median": median_transition,
                "ci_low": ci_low,
                "ci_high": ci_high,
            }

            # Plot if requested
            if save_plots:
                plt.figure(figsize=(10, 6))

                # Plot histogram of bootstrap transitions
                plt.hist(transitions, bins=20, alpha=0.7, color="skyblue")

                # Add vertical lines for mean, median, and confidence interval
                plt.axvline(
                    mean_transition,
                    color="red",
                    linestyle="-",
                    label=f"Mean: ${mean_transition:.2f}",
                )
                plt.axvline(
                    median_transition,
                    color="green",
                    linestyle="--",
                    label=f"Median: ${median_transition:.2f}",
                )
                plt.axvline(
                    ci_low,
                    color="purple",
                    linestyle=":",
                    label=f"95% CI: ${ci_low:.2f}",
                )
                plt.axvline(
                    ci_high, color="purple", linestyle=":", label=f"to ${ci_high:.2f}"
                )

                # Add labels and title
                plt.xlabel("Transition Value")
                plt.ylabel("Frequency")
                plt.title(f"Bootstrap Distribution for {q} (N={len(transitions)})")
                plt.legend()

                # Adjust x-axis for readable values
                if not args.get("log_scale_money", False):
                    plt.ticklabel_format(style="plain", axis="x")

                # Save figure
                if isinstance(args, dict):
                    plot_path = f'{args["output_dir"]}/{args["prompt_type"]}/{args["experiment_name"]}/bootstrap_{model_name}_{q}_{n_bootstrap}.png'
                    plt.tight_layout()
                    plt.savefig(plot_path, dpi=300)
                    print(f"Saved bootstrap plot for '{q}' to {plot_path}")
                else:
                    plt.tight_layout()
                    plt.show()
                plt.close()
        else:
            print(f"No valid bootstrap samples for quantity '{q}'")

    # Create a summary plot with all quantities
    if save_plots and bootstrap_results:
        plt.figure(figsize=(12, 8))

        quantities = list(bootstrap_results.keys())
        means = [bootstrap_results[q]["mean"] for q in quantities]
        ci_lows = [bootstrap_results[q]["ci_low"] for q in quantities]
        ci_highs = [bootstrap_results[q]["ci_high"] for q in quantities]

        # Error bars represent 95% confidence intervals
        y_pos = np.arange(len(quantities))
        error = [
            np.array(means) - np.array(ci_lows),
            np.array(ci_highs) - np.array(means),
        ]

        plt.errorbar(means, y_pos, xerr=error, fmt="o", capsize=5, color="blue")

        # Add quantity labels
        plt.yticks(y_pos, quantities)

        # Add labels and title
        plt.xlabel("Transition Value ($)")
        plt.title(f"Transition Values with 95% CI - {model_name}")

        # Adjust x-axis for readable values
        plt.ticklabel_format(style="plain", axis="x")

        # Add grid for better readability
        plt.grid(axis="x", linestyle="--", alpha=0.7)

        # Save figure
        if isinstance(args, dict):
            summary_path = f'{args["output_dir"]}/{args["prompt_type"]}/{args["experiment_name"]}/bootstrap_summary_{model_name}_{n_bootstrap}.png'
            plt.tight_layout()
            plt.savefig(summary_path, dpi=300)
            print(f"Saved bootstrap summary plot to {summary_path}")
        else:
            plt.tight_layout()
            plt.show()
        plt.close()

    # Save bootstrap results to file
    if isinstance(args, dict):
        results_path = f'{args["output_dir"]}/{args["prompt_type"]}/{args["experiment_name"]}/bootstrap_results_{model_name}_{n_bootstrap}.pkl'
        with open(results_path, "wb") as f:
            pickle.dump(bootstrap_results, f)
        print(f"Saved bootstrap results to {results_path}")

    return bootstrap_results

In [60]:
n_bootstrap = 30
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_30.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_30.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_30.pkl

Mean TP for BS n_30:  8.8563 
Median TP for BS n_30:  8.7642


In [61]:
bs_paht = "results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_30.pkl"

with open(bs_paht, "rb") as f:
    bs_data = pickle.load(f)

bs_data

{np.float64(5.0): {'transitions': array([9.39281626, 9.51370278, 9.12831574, 9.11790376, 8.21239203,
         8.73362994, 8.53521782, 8.51960699, 9.01421935, 9.53396209,
         8.14592071, 8.48016058, 9.15499262, 8.16480537, 9.61581928,
         8.20188845, 8.61113943, 9.64239838, 8.79470551, 9.63091298,
         7.63277142, 9.15254558, 8.59336781, 8.49719401, 8.70911096,
         9.73762999, 9.88908859, 9.05546925, 7.77222056, 8.50538704]),
  'mean': np.float64(8.85630984209544),
  'median': np.float64(8.764167726214009),
  'ci_low': np.float64(7.733872044360882),
  'ci_high': np.float64(9.779281105588543)}}

In [62]:
n_bootstrap = 20
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_20.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_20.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_20.pkl

Mean TP for BS n_20:  8.9072 
Median TP for BS n_20:  8.9045


In [63]:
n_bootstrap = 50
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_50.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_50.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_50.pkl

Mean TP for BS n_50:  8.7173 
Median TP for BS n_50:  8.6426


In [64]:
n_bootstrap = 100
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_100.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_100.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_100.pkl

Mean TP for BS n_100:  8.6502 
Median TP for BS n_100:  8.6093


In [68]:
n_bootstrap = 200
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_200.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_200.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_200.pkl

Mean TP for BS n_200:  8.599 
Median TP for BS n_200:  8.5917


In [67]:
n_bootstrap = 500
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_500.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_500.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_500.pkl

Mean TP for BS n_500:  8.5925 
Median TP for BS n_500:  8.568


In [65]:
n_bootstrap = 1000
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_1000.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_1000.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_1000.pkl

Mean TP for BS n_1000:  8.5771 
Median TP for BS n_1000:  8.5641


In [66]:
n_bootstrap = 2000
bootstrap_results = {}
bootstrap_results[f"bs_distance_claude_{n_bootstrap}"] = bootstrap_LR_analysis(
    args, data, "claude3_5", n_bootstrap=n_bootstrap
)

print(
    f"\nMean TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)]["mean"],
        4,
    ),
    f"\nMedian TP for BS n_{n_bootstrap}: ",
    round(
        bootstrap_results[f"bs_distance_claude_{n_bootstrap}"][np.float64(5.0)][
            "median"
        ],
        4,
    ),
)

Saved bootstrap plot for '5.0' to results_bootstrap/distance/bootstrap/bootstrap_claude3_5_5.0_2000.png
Saved bootstrap summary plot to results_bootstrap/distance/bootstrap/bootstrap_summary_claude3_5_2000.png
Saved bootstrap results to results_bootstrap/distance/bootstrap/bootstrap_results_claude3_5_2000.pkl

Mean TP for BS n_2000:  8.5654 
Median TP for BS n_2000:  8.5568
